# 🏏 Indian Cricket Team — AI Player Selection System
### Pure Groq API · No CrewAI · No LiteLLM · No OpenAI
> **Data Reference**: [BCCI Official Website](https://www.bcci.tv/)  
> **LLM**: Groq (LLaMA 3 · Mixtral · Gemma — 100% Open Source)


## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install -q groq gradio plotly pandas numpy
!pip uninstall -q -y openai litellm crewai 2>/dev/null || true
print('Done — clean install, no OpenAI / LiteLLM / CrewAI.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 6.2 MB/s eta 0:00:00
Done — clean install, no OpenAI / LiteLLM / CrewAI.


## 📚 Step 2: Import Libraries

In [ ]:
import os, json, time
from datetime import datetime
from typing import List

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from groq import Groq
import gradio as gr

print(f'Imports OK — {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('Backend: Pure Groq SDK — no litellm, no openai, no crewai')


Imports OK — 2026-03-15 06:02:35
Backend: Pure Groq SDK — no litellm, no openai, no crewai


## 🔑 Step 3: Groq API Key

In [2]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('Key loaded from Colab Secrets!')
except Exception:
    GROQ_API_KEY = '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'  # replace with your key
    print('Paste your Groq key above. Get one free at https://console.groq.com')

try:
    _c = Groq(api_key=GROQ_API_KEY)
    _r = _c.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role':'user','content':'Reply: Groq Connected!'}],
        max_tokens=10,
    )
    print(f'Groq API OK: {_r.choices[0].message.content.strip()}')
except Exception as e:
    print(f'API Error: {e}')


Paste your Groq key above. Get one free at https://console.groq.com
API Error: name 'Groq' is not defined


## 🤖 Step 4: Select Open-Source Model

In [ ]:
GROQ_MODELS = {
    'llama3-70b-8192':       'Meta LLaMA 3 70B   — Best quality (Recommended)',
    'llama3-8b-8192':        'Meta LLaMA 3 8B    — Fastest',
    'mixtral-8x7b-32768':    'Mistral Mixtral 8x7B — 32K context',
    'gemma2-9b-it':          'Google Gemma 2 9B  — Balanced',
    'llama-3.1-8b-instant':  'Meta LLaMA 3.1 8B  — Ultra-fast',
}
print('Available Models:')
for mid, desc in GROQ_MODELS.items():
    print(f'  {desc}  [{mid}]')

SELECTED_MODEL = 'llama-3.1-8b-instant'   # change here if needed

groq_client = Groq(api_key=GROQ_API_KEY)

def call_groq(system_prompt: str, user_prompt: str,
               max_tokens: int = 1800,
               temperature: float = 0.3) -> str:
    """Single reusable Groq API call — used by every agent."""
    try:
        resp = groq_client.chat.completions.create(
            model=SELECTED_MODEL,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f'[Groq Error]: {e}'

print(f'\nSelected: {SELECTED_MODEL}')
print('call_groq() ready — pure SDK, no wrappers.')


Available Models:
  Meta LLaMA 3 70B   — Best quality (Recommended)  [llama3-70b-8192]
  Meta LLaMA 3 8B    — Fastest  [llama3-8b-8192]
  Mistral Mixtral 8x7B — 32K context  [mixtral-8x7b-32768]
  Google Gemma 2 9B  — Balanced  [gemma2-9b-it]
  Meta LLaMA 3.1 8B  — Ultra-fast  [llama-3.1-8b-instant]

Selected: llama-3.1-8b-instant
call_groq() ready — pure SDK, no wrappers.


## 🌐 Step 5: Player Database (BCCI Reference)

In [ ]:
# ── Every stat is accessed via .get(key, default) — no KeyError possible ──
BCCI_URL = 'https://www.bcci.tv/'

PLAYERS_DB = {
    # ─── BATTERS ───────────────────────────────────────────────────────────
    'Rohit Sharma': {
        'role':'Batter', 'bat':'Right-hand', 'bowl':'Right-arm off-break',
        'test':{'m':67,  'runs':4301,  'avg':40.57, 'sr':58.2,  'h100':12, 'h50':16, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':264, 'runs':10709, 'avg':49.58, 'sr':90.6,  'h100':31, 'h50':59, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':159, 'runs':4231,  'avg':32.05, 'sr':140.9, 'h100':5,  'h50':32, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Excellent', 'fitness':100, 'age':37, 'ipl':'Mumbai Indians'},
    'Virat Kohli': {
        'role':'Batter', 'bat':'Right-hand', 'bowl':'Right-arm medium',
        'test':{'m':123, 'runs':9230,  'avg':48.57, 'sr':56.0,  'h100':30, 'h50':31, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':295, 'runs':13906, 'avg':57.32, 'sr':93.5,  'h100':51, 'h50':72, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':125, 'runs':4188,  'avg':52.35, 'sr':137.8, 'h100':1,  'h50':38, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Good', 'fitness':100, 'age':36, 'ipl':'Royal Challengers Bangalore'},
    'Shubman Gill': {
        'role':'Batter', 'bat':'Right-hand', 'bowl':'Right-arm off-break',
        'test':{'m':28, 'runs':1840, 'avg':38.33, 'sr':57.8,  'h100':4, 'h50':9,  'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':58, 'runs':2756, 'avg':56.24, 'sr':100.7, 'h100':8, 'h50':14, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':31, 'runs':916,  'avg':34.07, 'sr':155.7, 'h100':2, 'h50':6,  'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Excellent', 'fitness':100, 'age':25, 'ipl':'Gujarat Titans'},
    'Yashasvi Jaiswal': {
        'role':'Batter', 'bat':'Left-hand', 'bowl':'Leg-break',
        'test':{'m':19, 'runs':1754, 'avg':57.82, 'sr':65.3,  'h100':5, 'h50':7, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':8,  'runs':312,  'avg':44.57, 'sr':97.2,  'h100':1, 'h50':1, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':22, 'runs':682,  'avg':42.62, 'sr':163.5, 'h100':1, 'h50':4, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Excellent', 'fitness':100, 'age':23, 'ipl':'Rajasthan Royals'},
    'KL Rahul': {
        'role':'Batter/WK', 'bat':'Right-hand', 'bowl':'-',
        'test':{'m':53, 'runs':2959, 'avg':35.65, 'sr':50.3,  'h100':7, 'h50':15, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':73, 'runs':2432, 'avg':45.88, 'sr':88.7,  'h100':5, 'h50':19, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':72, 'runs':2265, 'avg':37.09, 'sr':141.0, 'h100':2, 'h50':23, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Average', 'fitness':85, 'age':32, 'ipl':'Lucknow Super Giants'},
    'Suryakumar Yadav': {
        'role':'Batter', 'bat':'Right-hand', 'bowl':'Right-arm off-break',
        'test':{'m':7,  'runs':278,  'avg':27.80, 'sr':58.1,  'h100':0, 'h50':1,  'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':64, 'runs':2004, 'avg':42.64, 'sr':118.6, 'h100':3, 'h50':15, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':69, 'runs':2568, 'avg':48.45, 'sr':170.3, 'h100':4, 'h50':18, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Excellent', 'fitness':100, 'age':34, 'ipl':'Mumbai Indians'},
    # ─── WICKET-KEEPERS ────────────────────────────────────────────────────
    'Rishabh Pant': {
        'role':'WK-Batter', 'bat':'Left-hand', 'bowl':'-',
        'test':{'m':37, 'runs':2870, 'avg':44.84, 'sr':73.4,  'h100':5, 'h50':14, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':30, 'runs':865,  'avg':39.31, 'sr':103.2, 'h100':0, 'h50':5,  'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':66, 'runs':1364, 'avg':27.28, 'sr':126.8, 'h100':0, 'h50':6,  'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Excellent', 'fitness':90, 'age':27, 'ipl':'Delhi Capitals'},
    'Sanju Samson': {
        'role':'WK-Batter', 'bat':'Right-hand', 'bowl':'-',
        'test':{'m':2,  'runs':56,  'avg':28.0,  'sr':52.3,  'h100':0, 'h50':0, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'odi': {'m':33, 'runs':948, 'avg':34.57, 'sr':101.3, 'h100':2, 'h50':6, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        't20': {'m':34, 'runs':877, 'avg':35.08, 'sr':147.1, 'h100':3, 'h50':4, 'wkts':0, 'bavg':0.0, 'econ':0.0},
        'form':'Excellent', 'fitness':100, 'age':30, 'ipl':'Rajasthan Royals'},
    # ─── ALL-ROUNDERS ──────────────────────────────────────────────────────
    'Hardik Pandya': {
        'role':'All-Rounder', 'bat':'Right-hand', 'bowl':'Right-arm fast-medium',
        'test':{'m':11, 'runs':532,  'avg':31.29, 'sr':68.5,  'h100':0, 'h50':4,  'wkts':17,  'bavg':31.80, 'econ':3.4},
        'odi': {'m':80, 'runs':1846, 'avg':31.98, 'sr':120.8, 'h100':0, 'h50':14, 'wkts':85,  'bavg':38.20, 'econ':5.5},
        't20': {'m':88, 'runs':1350, 'avg':28.70, 'sr':147.9, 'h100':0, 'h50':6,  'wkts':77,  'bavg':25.30, 'econ':8.9},
        'form':'Good', 'fitness':75, 'age':31, 'ipl':'Mumbai Indians'},
    'Ravindra Jadeja': {
        'role':'All-Rounder', 'bat':'Left-hand', 'bowl':'Left-arm orthodox',
        'test':{'m':80,  'runs':3175, 'avg':35.27, 'sr':59.2,  'h100':3, 'h50':18, 'wkts':295, 'bavg':24.07, 'econ':2.6},
        'odi': {'m':196, 'runs':2756, 'avg':32.01, 'sr':87.9,  'h100':0, 'h50':14, 'wkts':220, 'bavg':36.65, 'econ':4.9},
        't20': {'m':74,  'runs':515,  'avg':22.39, 'sr':127.2, 'h100':0, 'h50':1,  'wkts':54,  'bavg':29.60, 'econ':7.6},
        'form':'Excellent', 'fitness':95, 'age':36, 'ipl':'Chennai Super Kings'},
    'Axar Patel': {
        'role':'All-Rounder', 'bat':'Left-hand', 'bowl':'Left-arm orthodox',
        'test':{'m':28, 'runs':850, 'avg':28.33, 'sr':62.5,  'h100':1, 'h50':4, 'wkts':99, 'bavg':25.45, 'econ':2.8},
        'odi': {'m':69, 'runs':658, 'avg':27.41, 'sr':103.6, 'h100':0, 'h50':3, 'wkts':70, 'bavg':33.50, 'econ':5.1},
        't20': {'m':65, 'runs':553, 'avg':27.65, 'sr':154.9, 'h100':0, 'h50':1, 'wkts':74, 'bavg':19.80, 'econ':7.8},
        'form':'Excellent', 'fitness':100, 'age':31, 'ipl':'Delhi Capitals'},
    # ─── FAST BOWLERS ──────────────────────────────────────────────────────
    'Jasprit Bumrah': {
        'role':'Fast Bowler', 'bat':'Right-hand', 'bowl':'Right-arm fast',
        'test':{'m':45, 'runs':50,  'avg':5.0,  'sr':20.0, 'h100':0, 'h50':0, 'wkts':195, 'bavg':19.53, 'econ':2.7},
        'odi': {'m':97, 'runs':60,  'avg':4.0,  'sr':25.0, 'h100':0, 'h50':0, 'wkts':158, 'bavg':24.51, 'econ':4.6},
        't20': {'m':85, 'runs':30,  'avg':3.0,  'sr':20.0, 'h100':0, 'h50':0, 'wkts':89,  'bavg':20.22, 'econ':6.2},
        'form':'Excellent', 'fitness':90, 'age':31, 'ipl':'Mumbai Indians'},
    'Mohammed Shami': {
        'role':'Fast Bowler', 'bat':'Right-hand', 'bowl':'Right-arm fast-medium',
        'test':{'m':64,  'runs':80,  'avg':5.0,  'sr':22.0, 'h100':0, 'h50':0, 'wkts':229, 'bavg':27.06, 'econ':3.1},
        'odi': {'m':101, 'runs':70,  'avg':4.5,  'sr':20.0, 'h100':0, 'h50':0, 'wkts':195, 'bavg':27.78, 'econ':5.5},
        't20': {'m':23,  'runs':20,  'avg':3.0,  'sr':18.0, 'h100':0, 'h50':0, 'wkts':24,  'bavg':28.12, 'econ':8.4},
        'form':'Good', 'fitness':80, 'age':34, 'ipl':'Gujarat Titans'},
    'Mohammed Siraj': {
        'role':'Fast Bowler', 'bat':'Right-hand', 'bowl':'Right-arm fast-medium',
        'test':{'m':37, 'runs':60,  'avg':4.0,  'sr':18.0, 'h100':0, 'h50':0, 'wkts':112, 'bavg':29.97, 'econ':3.5},
        'odi': {'m':47, 'runs':40,  'avg':3.5,  'sr':16.0, 'h100':0, 'h50':0, 'wkts':76,  'bavg':33.42, 'econ':5.6},
        't20': {'m':24, 'runs':20,  'avg':3.0,  'sr':15.0, 'h100':0, 'h50':0, 'wkts':23,  'bavg':30.17, 'econ':9.2},
        'form':'Good', 'fitness':100, 'age':31, 'ipl':'Royal Challengers Bangalore'},
    'Arshdeep Singh': {
        'role':'Fast Bowler', 'bat':'Left-hand', 'bowl':'Left-arm fast-medium',
        'test':{'m':4,  'runs':15,  'avg':3.0,  'sr':14.0, 'h100':0, 'h50':0, 'wkts':11, 'bavg':35.54, 'econ':3.8},
        'odi': {'m':56, 'runs':50,  'avg':4.0,  'sr':18.0, 'h100':0, 'h50':0, 'wkts':80, 'bavg':29.92, 'econ':5.5},
        't20': {'m':65, 'runs':45,  'avg':4.0,  'sr':17.0, 'h100':0, 'h50':0, 'wkts':92, 'bavg':22.19, 'econ':8.2},
        'form':'Excellent', 'fitness':100, 'age':25, 'ipl':'Punjab Kings'},
    # ─── SPINNERS ──────────────────────────────────────────────────────────
    'Ravichandran Ashwin': {
        'role':'Spinner', 'bat':'Right-hand', 'bowl':'Right-arm off-break',
        'test':{'m':106, 'runs':3503, 'avg':26.74, 'sr':55.0, 'h100':6, 'h50':14, 'wkts':537, 'bavg':23.98, 'econ':2.7},
        'odi': {'m':116, 'runs':600,  'avg':15.0,  'sr':70.0, 'h100':0, 'h50':1,  'wkts':156, 'bavg':33.10, 'econ':4.9},
        't20': {'m':65,  'runs':200,  'avg':12.0,  'sr':95.0, 'h100':0, 'h50':0,  'wkts':72,  'bavg':23.60, 'econ':6.9},
        'form':'Good', 'fitness':100, 'age':38, 'ipl':'Chennai Super Kings'},
    'Kuldeep Yadav': {
        'role':'Spinner', 'bat':'Left-hand', 'bowl':'Left-arm wrist-spin',
        'test':{'m':19,  'runs':150, 'avg':10.0, 'sr':50.0, 'h100':0, 'h50':0, 'wkts':72,  'bavg':26.93, 'econ':3.1},
        'odi': {'m':100, 'runs':250, 'avg':9.0,  'sr':55.0, 'h100':0, 'h50':0, 'wkts':182, 'bavg':26.18, 'econ':5.1},
        't20': {'m':29,  'runs':60,  'avg':7.0,  'sr':70.0, 'h100':0, 'h50':0, 'wkts':42,  'bavg':15.10, 'econ':7.5},
        'form':'Excellent', 'fitness':100, 'age':30, 'ipl':'Delhi Capitals'},
}

VENUES = [
    'Wankhede Stadium, Mumbai','Eden Gardens, Kolkata',
    'M. Chinnaswamy Stadium, Bengaluru','Narendra Modi Stadium, Ahmedabad',
    'Arun Jaitley Stadium, Delhi','MA Chidambaram Stadium, Chennai',
    'Rajiv Gandhi International, Hyderabad','PCA Stadium, Mohali',
    "Lord's Cricket Ground, London",'MCG, Melbourne',
    'SCG, Sydney','The Oval, London','Headingley, Leeds',
]
OPPOSITION = [
    'Australia','England','South Africa','New Zealand',
    'Pakistan','West Indies','Sri Lanka','Bangladesh',
    'Afghanistan','Zimbabwe',
]

# ── Quick self-test: confirm no KeyError on any player/format ──
SAFE_KEYS = ['m','runs','avg','sr','h100','h50','wkts','bavg','econ']
errors = []
for name, p in PLAYERS_DB.items():
    for fmt in ['test','odi','t20']:
        s = p.get(fmt, {})
        for k in SAFE_KEYS:
            _ = s.get(k, 0)   # should never raise
if errors:
    print(f'Errors: {errors}')
else:
    print(f'Player DB OK — {len(PLAYERS_DB)} players, all stat keys validated.')
print(f'Venues: {len(VENUES)} | Opposition: {len(OPPOSITION)} | Source: {BCCI_URL}')


Player DB OK — 17 players, all stat keys validated.
Venues: 13 | Opposition: 10 | Source: https://www.bcci.tv/


## 🧠 Step 6: 5-Agent Pipeline (Pure Groq API)

In [ ]:
# ── Safe stat builder — uses .get(key, default) everywhere ──────────────
def build_player_table(players: List[str], fmt: str) -> str:
    lines = []
    for name in players:
        p = PLAYERS_DB.get(name)
        if not p:
            continue
        s = p.get(fmt, {})
        # Safe reads — default 0 for every stat
        m     = s.get('m',    0)
        runs  = s.get('runs', 0)
        avg   = s.get('avg',  0.0)
        sr    = s.get('sr',   0.0)
        h100  = s.get('h100', 0)
        h50   = s.get('h50',  0)
        wkts  = s.get('wkts', 0)
        bavg  = s.get('bavg', 0.0)
        econ  = s.get('econ', 0.0)
        fitness = p.get('fitness', 0)
        form    = p.get('form', 'Unknown')
        role    = p.get('role', 'Player')

        line = (
            f'{name} | {role} | '
            f'Matches:{m} | Form:{form} | Fitness:{fitness}%'
        )
        if runs > 0:
            line += f' | Runs:{runs} Avg:{avg} SR:{sr} 100s:{h100} 50s:{h50}'
        if wkts > 0:
            line += f' | Wkts:{wkts} BowlAvg:{bavg} Econ:{econ}'
        lines.append(line)
    return '\n'.join(lines)


# ── AGENT 1: Performance Analyst ─────────────────────────────────────────
def agent1_performance(players, fmt, format_type, opposition, venue):
    table = build_player_table(players, fmt)
    sys_p = (
        'You are a Senior BCCI Cricket Performance Analyst. '
        'You use bcci.tv, ESPNcricinfo, and Cricbuzz data. '
        'You rank players by their statistics, form, and head-to-head records. '
        'Be concise and specific.'
    )
    usr_p = (
        f'Rank all players for {format_type} vs {opposition} at {venue}.\n\n'
        f'PLAYER DATA:\n{table}\n\n'
        f'OUTPUT (numbered list):\n'
        f'- Score each player /100 based on {format_type} stats + form + fitness\n'
        f'- One-line justification per player\n'
        f'- Identify best openers, middle-order, finishers, WK, bowlers'
    )
    print('  [Agent 1] Performance Analyst...')
    return call_groq(sys_p, usr_p, max_tokens=1000)


# ── AGENT 2: Conditions Specialist ───────────────────────────────────────
def agent2_conditions(venue, pitch, weather, opposition):
    sys_p = (
        'You are a Pitch and Playing Conditions Specialist with 25 years experience. '
        'You have curated surfaces at 50+ international venues.'
    )
    usr_p = (
        f'Assess conditions for India vs {opposition}:\n'
        f'Venue: {venue} | Pitch: {pitch} | Weather: {weather}\n\n'
        f'Provide:\n'
        f'1. Pitch behaviour (bounce, swing, spin, pace rating /10)\n'
        f'2. Ideal batting profile for these conditions\n'
        f'3. Ideal bowling profile for these conditions\n'
        f'4. Toss recommendation — bat or bowl, with reason\n'
        f'5. Weather impact on the match (dew, cloud cover etc.)'
    )
    print('  [Agent 2] Conditions Specialist...')
    return call_groq(sys_p, usr_p, max_tokens=700)


# ── AGENT 3: Fitness Monitor ──────────────────────────────────────────────
def agent3_fitness(players, priority):
    lines = []
    for name in players:
        p = PLAYERS_DB.get(name)
        if not p:
            continue
        fit = p.get('fitness', 0)
        tag = 'GREEN' if fit >= 90 else ('AMBER' if fit >= 75 else 'RED')
        lines.append(
            f'{name} | {p.get("role","")} | '
            f'Fitness:{fit}% | Form:{p.get("form","")} | Status:{tag}'
        )
    table = '\n'.join(lines)
    sys_p = (
        'You are the Head of BCCI Sports Science and Medical Division. '
        'You track every player injury status, workload, and physical readiness.'
    )
    usr_p = (
        f'Review fitness and availability. Priority: {priority}\n\n'
        f'FITNESS DATA:\n{table}\n\n'
        f'Provide:\n'
        f'1. RED players (below 80%) — flag as high-risk, recommend rest\n'
        f'2. AMBER players (80-89%) — use with caution, list\n'
        f'3. GREEN players (90%+) — fully available, list\n'
        f'4. Workload management notes\n'
        f'5. Final unavailability recommendation'
    )
    print('  [Agent 3] Fitness Monitor...')
    return call_groq(sys_p, usr_p, max_tokens=700)


# ── AGENT 4: Team Strategist ──────────────────────────────────────────────
def agent4_strategist(players, fmt, format_type, strategy, perf, cond):
    table = build_player_table(players, fmt)
    sys_p = (
        'You are a former Indian cricket captain and Team Balance Architect. '
        'You design perfectly balanced XIs with ideal batting orders and bowling plans.'
    )
    usr_p = (
        f'Propose a balanced Playing XI for {format_type}. Strategy: {strategy}\n\n'
        f'PLAYERS:\n{table}\n\n'
        f'PERFORMANCE SUMMARY:\n{perf[:500]}\n\n'
        f'CONDITIONS SUMMARY:\n{cond[:350]}\n\n'
        f'Provide:\n'
        f'1. Playing XI (11 players, roles)\n'
        f'2. Batting order (positions 1-11)\n'
        f'3. Bowling combination (overs allocation)\n'
        f'4. Team balance check (batters/AR/WK/bowlers count)'
    )
    print('  [Agent 4] Team Strategist...')
    return call_groq(sys_p, usr_p, max_tokens=900)


# ── AGENT 5: Chief Selector ───────────────────────────────────────────────
def agent5_chief_selector(
        format_type, opposition, venue, pitch, weather,
        strategy, priority, perf, cond, fit, strat):
    sys_p = (
        'You are the BCCI Chief Selector with 30 years of Indian cricket experience. '
        'You synthesise all expert inputs and make the final, authoritative team announcement. '
        'You are data-driven, fair, and always explain your decisions clearly.'
    )
    usr_p = (
        f'FINAL SELECTION — India vs {opposition} | {format_type} | {venue}\n'
        f'Pitch: {pitch} | Weather: {weather} | '
        f'Strategy: {strategy} | Priority: {priority}\n\n'
        f'=== PERFORMANCE REPORT ===\n{perf[:600]}\n\n'
        f'=== CONDITIONS REPORT ===\n{cond[:400]}\n\n'
        f'=== FITNESS REPORT ===\n{fit[:400]}\n\n'
        f'=== STRATEGY PROPOSAL ===\n{strat[:500]}\n\n'
        f'Announce the final team with ALL sections below:\n'
        f'1. PLAYING XI — numbered 1-11 with batting position and role\n'
        f'2. CAPTAIN — name + reason\n'
        f'3. VICE-CAPTAIN — name + reason\n'
        f'4. BATTING ORDER — top / middle / lower order breakdown\n'
        f'5. BOWLING PLAN — main bowlers and overs allocation\n'
        f'6. FOUR RESERVES — names with roles\n'
        f'7. TEAM STRENGTH SCORE — X/100 with brief breakdown\n'
        f'8. WIN PROBABILITY vs {opposition} — % with reasoning\n'
        f'9. KEY ADVANTAGES — top 3 edges India has\n'
        f'10. CONCERNS — top 2 risks to manage\n'
        f'11. MATCH PREDICTION — concise 3-sentence gameplan'
    )
    print('  [Agent 5] Chief Selector...')
    return call_groq(sys_p, usr_p, max_tokens=1800)


# ── ORCHESTRATOR ─────────────────────────────────────────────────────────
def run_pipeline(
        format_type, opposition, venue, pitch, weather,
        priority, strategy, player_pool, progress=None):

    # Validate pool
    players = [p for p in player_pool if p in PLAYERS_DB]
    if len(players) < 11:
        return (f'Need at least 11 valid players. Currently selected: {len(players)}',
                'Error — insufficient players')

    fmt = 't20' if 'T20' in format_type else ('odi' if 'ODI' in format_type else 'test')

    def upd(pct, msg):
        if progress:
            progress(pct, desc=msg)
        print(f'[{int(pct*100):3d}%] {msg}')

    try:
        upd(0.08, 'Agent 1 — Performance Analyst ranking players...')
        perf = agent1_performance(players, fmt, format_type, opposition, venue)
        time.sleep(0.4)

        upd(0.28, 'Agent 2 — Conditions Specialist assessing pitch & weather...')
        cond = agent2_conditions(venue, pitch, weather, opposition)
        time.sleep(0.4)

        upd(0.48, 'Agent 3 — Fitness Monitor reviewing player readiness...')
        fit  = agent3_fitness(players, priority)
        time.sleep(0.4)

        upd(0.66, 'Agent 4 — Team Strategist proposing balanced XI...')
        strat = agent4_strategist(players, fmt, format_type, strategy, perf, cond)
        time.sleep(0.4)

        upd(0.84, 'Agent 5 — Chief Selector announcing final XI...')
        final = agent5_chief_selector(
            format_type, opposition, venue, pitch, weather,
            strategy, priority, perf, cond, fit, strat)

        upd(1.00, f'Complete! {len(players)} players analysed by 5 agents.')

        report = (
            f'## India vs {opposition} | {format_type} | {venue}\n'
            f'**Pitch:** {pitch} &nbsp;|&nbsp; **Weather:** {weather} '
            f'&nbsp;|&nbsp; **Strategy:** {strategy} '
            f'&nbsp;|&nbsp; **Priority:** {priority}\n'
            f'**Players analysed:** {len(players)} '
            f'&nbsp;|&nbsp; **Model:** {SELECTED_MODEL} (Groq)\n\n'
            f'---\n\n'
            f'### Agent 1 — Performance Analysis\n\n{perf}\n\n'
            f'---\n\n'
            f'### Agent 2 — Conditions Report\n\n{cond}\n\n'
            f'---\n\n'
            f'### Agent 3 — Fitness Report\n\n{fit}\n\n'
            f'---\n\n'
            f'### Agent 4 — Strategy Proposal\n\n{strat}\n\n'
            f'---\n\n'
            f'## Chief Selector — Final Announcement\n\n{final}\n\n'
            f'---\n'
            f'*Source: {BCCI_URL} | Model: {SELECTED_MODEL} via Groq*'
        )
        return report, f'Done! {len(players)} players analysed.'

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return f'Pipeline Error:\n{tb}', 'Failed'


print('5-Agent pipeline ready!')
print('Agents: Performance | Conditions | Fitness | Strategist | Chief Selector')
print('No KeyError possible — all stats use .get(key, default).')


5-Agent pipeline ready!
Agents: Performance | Conditions | Fitness | Strategist | Chief Selector
No KeyError possible — all stats use .get(key, default).


## 🎨 Step 7: Gradio UI

In [ ]:
# ── UI helper: player profile card ──────────────────────────────────────
def get_fmt(fmt):
    return 't20' if 'T20' in fmt else ('odi' if 'ODI' in fmt else 'test')


def player_card(name, fmt):
    p = PLAYERS_DB.get(name)
    if not p:
        return '<p style="color:red">Player not found.</p>'
    s  = p.get(get_fmt(fmt), {})
    fc = {'Excellent':'#2e7d32','Good':'#1565c0','Average':'#e65100','Poor':'#b71c1c'}
    fit = p.get('fitness', 0)
    fc2 = '#2e7d32' if fit>=90 else ('#e65100' if fit>=75 else '#b71c1c')
    rows = ''
    pairs = [
        ('m','Matches'),('runs','Runs'),('avg','Batting Avg'),
        ('sr','Strike Rate'),('h100','Centuries'),('h50','Half-Centuries'),
        ('wkts','Wickets'),('bavg','Bowl Avg'),('econ','Economy'),
    ]
    for k, lbl in pairs:
        val = s.get(k, 0)
        if val and val != 0:
            rows += (
                f'<tr style="border-bottom:1px solid #eef2ff;">'
                f'<td style="padding:8px 14px;color:#546e7a;">{lbl}</td>'
                f'<td style="padding:8px 14px;font-weight:700;color:#1a237e;">{val}</td>'
                f'</tr>'
            )
    return (
        f'<div style="font-family:Georgia,serif;'
        f'background:linear-gradient(135deg,#e8f5e9,#f0f4ff);'
        f'border-radius:16px;padding:24px;border:2px solid #4caf50;'
        f'box-shadow:0 4px 20px rgba(26,35,126,.1);margin:8px 0;">'
        f'<h3 style="color:#1a237e;margin:0 0 14px;font-size:1.4em;">🏏 {name}</h3>'
        f'<div style="display:flex;gap:9px;flex-wrap:wrap;margin-bottom:16px;">'
        f'<span style="background:#1a237e;color:#fff;padding:5px 13px;border-radius:20px;font-size:.83em;">{p["role"]}</span>'
        f'<span style="background:{fc.get(p["form"],"#555")};color:#fff;padding:5px 13px;border-radius:20px;font-size:.83em;">Form: {p["form"]}</span>'
        f'<span style="background:{fc2};color:#fff;padding:5px 13px;border-radius:20px;font-size:.83em;">Fitness: {fit}%</span>'
        f'<span style="background:#37474f;color:#fff;padding:5px 13px;border-radius:20px;font-size:.83em;">Age {p["age"]}</span>'
        f'</div>'
        f'<table style="width:100%;border-collapse:collapse;font-size:.92em;">'
        f'<thead><tr style="background:#1a237e;color:#fff;">'
        f'<th style="padding:9px 14px;text-align:left;">Statistic</th>'
        f'<th style="padding:9px 14px;text-align:left;">{fmt}</th></tr></thead>'
        f'<tbody>{rows}</tbody></table>'
        f'<p style="margin:13px 0 0;color:#78909c;font-size:.81em;">'
        f'IPL: {p.get("ipl","N/A")} | Bat: {p["bat"]} | Bowl: {p["bowl"]} | bcci.tv</p>'
        f'</div>'
    )


# ── UI helper: radar chart ────────────────────────────────────────────────
def radar_chart(players, fmt):
    fk   = get_fmt(fmt)
    cats = ['Bat Avg','Strike Rate','Experience','Form','Fitness']
    fig  = go.Figure()
    fmap = {'Excellent':95,'Good':75,'Average':50,'Poor':25}
    cols = px.colors.qualitative.Set2
    for i, name in enumerate(players[:6]):
        p = PLAYERS_DB.get(name)
        if not p: continue
        s = p.get(fk, {})
        vals = [
            min(s.get('avg', 0), 70) / 70 * 100,
            min(s.get('sr',  0), 200) / 200 * 100,
            min(s.get('m',   0), 150) / 150 * 100,
            fmap.get(p.get('form','Average'), 50),
            p.get('fitness', 0),
        ]
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=cats, fill='toself', name=name,
            line_color=cols[i % len(cols)], opacity=0.70))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0,100],
            gridcolor='rgba(26,35,126,.12)')),
        title=dict(text=f'Player Comparison — {fmt}',
            font=dict(size=15,color='#1a237e',family='Georgia')),
        paper_bgcolor='rgba(232,245,233,.35)',
        height=430, font=dict(family='Georgia'),
        margin=dict(t=65,b=25,l=25,r=25))
    return fig


# ── UI helper: bar chart ──────────────────────────────────────────────────
def bar_chart(players, fmt):
    fk = get_fmt(fmt)
    names, avgs, srs = [], [], []
    for name in players:
        p = PLAYERS_DB.get(name)
        if not p: continue
        s = p.get(fk, {})
        avg_v = s.get('avg', 0)
        sr_v  = s.get('sr',  0)
        if avg_v > 0:
            names.append(name.split()[-1])
            avgs.append(round(avg_v, 1))
            srs.append(round(sr_v, 1))
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=('Batting Average','Strike Rate'),
        horizontal_spacing=0.14)
    fig.add_trace(go.Bar(x=names, y=avgs,
        marker_color='#1a237e', text=avgs, textposition='outside',
        name='Avg'), row=1, col=1)
    fig.add_trace(go.Bar(x=names, y=srs,
        marker_color='#43a047', text=srs, textposition='outside',
        name='SR'), row=1, col=2)
    fig.update_layout(
        title=dict(text=f'Batting Stats — {fmt}',
            font=dict(size=14,color='#1a237e',family='Georgia')),
        paper_bgcolor='rgba(232,245,233,.35)',
        plot_bgcolor='rgba(255,255,255,.6)',
        height=370, showlegend=False,
        font=dict(family='Georgia'),
        margin=dict(t=55,b=30,l=10,r=10))
    fig.update_xaxes(tickangle=-30)
    return fig


# ── Quick AI ──────────────────────────────────────────────────────────────
def quick_ai(question):
    if not question.strip():
        return 'Please enter a question.'
    roster = {
        k: {'role':v['role'],'form':v['form'],
             'fitness':str(v['fitness'])+'%','age':v['age']}
        for k,v in PLAYERS_DB.items()
    }
    sys_p = (
        'You are a senior BCCI cricket analyst. '
        f'Current squad: {json.dumps(roster)}. '
        'Give expert, data-driven answers. Cite bcci.tv.'
    )
    ans = call_groq(sys_p, question, max_tokens=1000)
    return f'### AI Cricket Analysis\n\n{ans}\n\n---\n*{SELECTED_MODEL} via Groq | bcci.tv*'


print('UI helpers ready!')


UI helpers ready!


In [ ]:
# ── Gradio CSS & HTML constants ──────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=Source+Sans+3:wght@400;600&display=swap');
body,.gradio-container{
  font-family:'Source Sans 3',sans-serif!important;
  background:linear-gradient(150deg,#f0f7ee 0%,#e8f4fd 55%,#fdf3f0 100%)!important;
}
.hdr{
  background:linear-gradient(135deg,#0d1b5e 0%,#1a237e 45%,#1565c0 100%);
  padding:28px 36px;border-radius:18px;text-align:center;
  box-shadow:0 10px 36px rgba(26,35,126,.32);margin-bottom:20px;
}
.hdr h1{
  font-family:'Playfair Display',serif!important;
  font-size:2.05em;color:#fff!important;margin:0;letter-spacing:.5px;
}
.hdr p{color:#f9c74f!important;margin:8px 0 0;font-size:.98em;}
.badges{display:flex;gap:8px;justify-content:center;margin-top:13px;flex-wrap:wrap;}
.bdg{
  background:rgba(255,255,255,.13);border:1px solid rgba(249,199,79,.45);
  color:#f9c74f;padding:3px 12px;border-radius:20px;font-size:.79em;
}
.sec{
  font-family:'Playfair Display',serif!important;
  color:#1a237e!important;font-size:1.06em!important;
  border-bottom:2px solid #f9a825;padding-bottom:7px;margin-bottom:13px!important;
}
.pipe{
  background:linear-gradient(135deg,#e8eaf6,#e3f2fd);
  border-radius:13px;padding:17px;
  border:1px solid rgba(26,35,126,.12);margin:12px 0;
}
.ps{display:flex;align-items:flex-start;gap:11px;margin-bottom:11px;}
.pn{
  color:white;width:28px;height:28px;border-radius:50%;
  display:flex;align-items:center;justify-content:center;
  font-weight:700;flex-shrink:0;font-size:.87em;
}
.note{
  background:linear-gradient(90deg,#fff8e1,#fffde7);
  border-left:4px solid #f9a825;border-radius:0 10px 10px 0;
  padding:10px 15px;margin:10px 0;font-size:.87em;color:#5d4037;
}
"""

HEADER = """
<div class="hdr">
  <div style="font-size:2.6em;margin-bottom:7px;">🏏</div>
  <h1>Indian Cricket Team — AI Selector</h1>
  <p>5-Agent Pipeline &nbsp;·&nbsp; Pure Groq API &nbsp;·&nbsp; No OpenAI &nbsp;·&nbsp; No LiteLLM</p>
  <div class="badges">
    <span class="bdg">5 AI Agents</span>
    <span class="bdg">BCCI Reference</span>
    <span class="bdg">LLaMA 3 · Mixtral · Gemma</span>
    <span class="bdg">Pure Groq API</span>
    <span class="bdg">No KeyErrors</span>
    <span class="bdg">bcci.tv</span>
  </div>
</div>
"""

PIPE_HTML = """
<div class="pipe">
  <h3 style="font-family:Georgia,serif;color:#1a237e;margin:0 0 13px;font-size:.97em;">Multi-Agent Pipeline</h3>
  <div class="ps"><div class="pn" style="background:#1a237e;">1</div>
    <div><b style="color:#1a237e;">Performance Analyst</b><br>
    <span style="font-size:.81em;color:#555;">Stats ranking, career records, h2h</span></div></div>
  <div class="ps"><div class="pn" style="background:#2e7d32;">2</div>
    <div><b style="color:#2e7d32;">Conditions Specialist</b><br>
    <span style="font-size:.81em;color:#555;">Pitch, weather, venue factors</span></div></div>
  <div class="ps"><div class="pn" style="background:#e65100;">3</div>
    <div><b style="color:#e65100;">Fitness Monitor</b><br>
    <span style="font-size:.81em;color:#555;">Injury, workload, form rating</span></div></div>
  <div class="ps"><div class="pn" style="background:#6a1b9a;">4</div>
    <div><b style="color:#6a1b9a;">Team Strategist</b><br>
    <span style="font-size:.81em;color:#555;">Balance, batting order, bowling plan</span></div></div>
  <div class="ps" style="margin-bottom:0;"><div class="pn" style="background:#f9a825;color:#1a237e;">5</div>
    <div><b style="color:#b8600a;">Chief Selector</b><br>
    <span style="font-size:.81em;color:#555;">Final XI, captain, reserves, prediction</span></div></div>
</div>
"""

NOTE_HTML = """
<div class="note">
  No OpenAI key needed &nbsp;·&nbsp; No LiteLLM &nbsp;·&nbsp; No CrewAI &nbsp;·&nbsp;
  Pure <b>Groq SDK</b> only &nbsp;·&nbsp; All stats use <code>.get(key, 0)</code> — no KeyErrors.
</div>
"""

# ── Build Gradio app ──────────────────────────────────────────────────────
all_players = list(PLAYERS_DB.keys())

with gr.Blocks(css=CSS, title='India Cricket AI Selector',
               theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate')) as app:

    gr.HTML(HEADER)

    with gr.Tabs():

        # ── TAB 1 : TEAM SELECTOR ────────────────────────────────────
        with gr.TabItem('🏏 Team Selector'):
            gr.HTML(NOTE_HTML)
            with gr.Row():

                # LEFT — config
                with gr.Column(scale=1, min_width=285):
                    gr.HTML('<div class="sec">Match Configuration</div>')
                    fmt_dd = gr.Dropdown(
                        ['Test Match','ODI (50 Overs)','T20 International'],
                        value='T20 International', label='Match Format')
                    opp_dd = gr.Dropdown(
                        OPPOSITION, value='Australia', label='Opposition Team')
                    ven_dd = gr.Dropdown(
                        VENUES, value='Wankhede Stadium, Mumbai', label='Match Venue')
                    pit_dd = gr.Dropdown(
                        ['Batting Pitch (Flat)','Seaming Pitch (Green)',
                         'Spin-Friendly (Dry)','Balanced Pitch',
                         'Bouncy Pitch (Pace)','Dead Wicket'],
                        value='Balanced Pitch', label='Pitch Type')
                    wea_dd = gr.Dropdown(
                        ['Sunny and Dry','Overcast and Humid','Windy',
                         'Light Drizzle (DLS Possible)','Hot and Dry',
                         'Clear Evening (Dew Factor)'],
                        value='Clear Evening (Dew Factor)', label='Weather')
                    gr.HTML('<div class="sec" style="margin-top:13px;">Strategy Settings</div>')
                    str_dd = gr.Dropdown(
                        ['Aggressive (High-Risk High-Reward)',
                         'Conservative (Safety First)',
                         'Balanced (All-Round Team)',
                         'Pace-Heavy Attack','Spin Dominance',
                         'Batting Powerhouse','Youth Development',
                         'Series Decider (Best XI)'],
                        value='Balanced (All-Round Team)', label='Playing Strategy')
                    pri_dd = gr.Dropdown(
                        ['Win at All Costs','Player Development',
                         'Experience Over Youth','Youth Over Experience',
                         'World Cup Preparation','Workload Management'],
                        value='Win at All Costs', label='Selection Priority')
                    gr.HTML(PIPE_HTML)

                # RIGHT — pool + results
                with gr.Column(scale=2):
                    gr.HTML('<div class="sec">Select Player Pool (min 11)</div>')
                    pool_cb = gr.CheckboxGroup(
                        choices=all_players, value=all_players[:15],
                        label='Available Players — BCCI Registered')
                    with gr.Row():
                        b_all = gr.Button('Select All',   size='sm', variant='secondary')
                        b_clr = gr.Button('Clear All',    size='sm', variant='secondary')
                        b_15  = gr.Button('Top 15',       size='sm', variant='secondary')
                        b_bat = gr.Button('Batters Only',  size='sm', variant='secondary')
                        b_bwl = gr.Button('Bowlers Only',  size='sm', variant='secondary')
                    b_all.click(lambda: all_players, outputs=pool_cb)
                    b_clr.click(lambda: [],          outputs=pool_cb)
                    b_15.click(lambda: all_players[:15], outputs=pool_cb)
                    b_bat.click(
                        lambda: [n for n,d in PLAYERS_DB.items()
                                 if 'Batter' in d['role'] or 'WK' in d['role']],
                        outputs=pool_cb)
                    b_bwl.click(
                        lambda: [n for n,d in PLAYERS_DB.items()
                                 if any(x in d['role'] for x in
                                        ['Bowler','Spinner','All-Rounder'])],
                        outputs=pool_cb)

                    run_btn = gr.Button(
                        'SELECT PLAYING XI — Run 5-Agent AI Analysis',
                        variant='primary', size='lg')
                    status_tb = gr.Textbox(
                        label='Pipeline Status',
                        value='Ready — configure settings and click Run',
                        interactive=False)
                    result_md = gr.Markdown(
                        '*Results appear here after the 5-agent analysis...*',
                        label='Team Selection Report')

            def do_run(fmt, opp, ven, pit, wea, pri, strat, pool,
                       prog=gr.Progress()):
                return run_pipeline(fmt, opp, ven, pit, wea, pri, strat, pool, prog)

            run_btn.click(
                fn=do_run,
                inputs=[fmt_dd, opp_dd, ven_dd, pit_dd,
                        wea_dd, pri_dd, str_dd, pool_cb],
                outputs=[result_md, status_tb])

        # ── TAB 2 : PLAYER PROFILES ──────────────────────────────────
        with gr.TabItem('👤 Player Profiles'):
            gr.HTML('<div class="sec">Individual Stats — BCCI Data Reference</div>')
            with gr.Row():
                p_dd  = gr.Dropdown(all_players, value=all_players[0], label='Player')
                pf_dd = gr.Dropdown(
                    ['Test Match','ODI (50 Overs)','T20 International'],
                    value='T20 International', label='Format')
                p_btn = gr.Button('Load Profile', variant='primary', size='sm')
            p_html = gr.HTML('<p style="color:#546e7a;padding:18px;">Select a player above.</p>')
            p_btn.click(fn=player_card,  inputs=[p_dd, pf_dd], outputs=p_html)
            p_dd.change(fn=player_card,  inputs=[p_dd, pf_dd], outputs=p_html)
            pf_dd.change(fn=player_card, inputs=[p_dd, pf_dd], outputs=p_html)

        # ── TAB 3 : COMPARISON ───────────────────────────────────────
        with gr.TabItem('⚔️ Compare Players'):
            gr.HTML('<div class="sec">Head-to-Head Comparison Charts</div>')
            with gr.Row():
                cmp_cb  = gr.CheckboxGroup(
                    all_players, value=all_players[:4],
                    label='Select Players (max 6)')
                cmp_fmt = gr.Dropdown(
                    ['Test Match','ODI (50 Overs)','T20 International'],
                    value='T20 International', label='Format')
            cmp_btn = gr.Button('Generate Charts', variant='primary')
            with gr.Row():
                r_plt = gr.Plot(label='Radar Comparison')
                b_plt = gr.Plot(label='Batting Bar Chart')
            cmp_btn.click(
                fn=lambda p, f: (radar_chart(p, f), bar_chart(p, f)),
                inputs=[cmp_cb, cmp_fmt], outputs=[r_plt, b_plt])

        # ── TAB 4 : QUICK AI ─────────────────────────────────────────
        with gr.TabItem('💬 Quick AI Analysis'):
            gr.HTML('<div class="sec">Ask Anything About Indian Cricket</div>')
            q_tmpl = gr.Dropdown(
                choices=[
                    'Who should open batting in T20 Internationals?',
                    'Best bowling combination for Australia in Tests?',
                    'Who is the best T20 captain for India?',
                    'Compare Bumrah vs Shami for ODI cricket',
                    'Who should be Indias T20 finisher at No 6 or 7?',
                    'Best spin duo for subcontinental pitch conditions?',
                    'Ideal batting order for a World Cup knockout match?',
                    'Who should India rest for workload management?',
                ],
                value='Who should open batting in T20 Internationals?',
                label='Quick Templates')
            q_txt = gr.Textbox(
                label='Your Question',
                value='Who should open batting in T20 Internationals?',
                lines=2, placeholder='Type your cricket question...')
            q_tmpl.change(fn=lambda t: t, inputs=q_tmpl, outputs=q_txt)
            q_btn = gr.Button('Get AI Analysis', variant='primary')
            q_out = gr.Markdown('*Click Get AI Analysis...*')
            q_btn.click(fn=quick_ai, inputs=q_txt, outputs=q_out)

        # ── TAB 5 : ABOUT ────────────────────────────────────────────
        with gr.TabItem('About'):
            gr.Markdown("""
## Indian Cricket AI Selector — 100% Open Source

### Bug Fixes Applied
| Error | Fix |
|-------|-----|
| `KeyError: '100s'` | Renamed to `h100`/`h50`; all reads use `.get(key, 0)` |
| `litellm` circular import | Removed CrewAI entirely |
| `OPENAI_API_KEY` required | Pure Groq SDK — no wrappers |

### Tech Stack
| Component | Choice |
|-----------|--------|
| LLM Calls | `groq` Python SDK (direct, no wrappers) |
| Models | Meta LLaMA 3 70B / Mistral Mixtral / Google Gemma |
| Agents | Custom Python functions (no crewai) |
| UI | Gradio 4.x |
| Charts | Plotly |
| Data Source | [BCCI (bcci.tv)](https://www.bcci.tv/) |

### Get Free Groq Key
1. Visit **https://console.groq.com**
2. Sign up (generous free tier)
3. Create API key
4. In Colab: 🔑 Secrets → add `GROQ_API_KEY`
            """)

    gr.HTML("""
    <div style="text-align:center;padding:16px;margin-top:16px;
         background:linear-gradient(90deg,rgba(26,35,126,.05),rgba(46,125,50,.05));
         border-radius:12px;border-top:3px solid #f9a825;">
      <p style="color:#1a237e;font-weight:700;margin:0;">Indian Cricket AI Selector</p>
      <p style="color:#666;font-size:.83em;margin:5px 0 0;">
        Pure Groq SDK &nbsp;·&nbsp; LLaMA 3 / Mixtral / Gemma &nbsp;·&nbsp;
        No OpenAI &nbsp;·&nbsp; No LiteLLM &nbsp;·&nbsp;
        Data: <a href="https://www.bcci.tv/" style="color:#1a237e;">bcci.tv</a>
      </p>
    </div>""")

print('Gradio UI built — no KeyErrors, no OpenAI, no LiteLLM.')


/tmp/ipykernel_949/1231680454.py:93: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='India Cricket AI Selector',
/tmp/ipykernel_949/1231680454.py:93: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='India Cricket AI Selector',


Gradio UI built — no KeyErrors, no OpenAI, no LiteLLM.


## 🚀 Step 8: Launch

In [ ]:
print('Launching India Cricket AI Selector...')
print(f'  Model  : {SELECTED_MODEL}')
print(f'  Engine : Pure Groq SDK — no openai/litellm/crewai')
print(f'  Agents : 5 custom agents')
print(f'  Players: {len(PLAYERS_DB)}')
print(f'  Source : https://www.bcci.tv/')

app.launch(share=True, debug=True, show_error=True,
            server_port=7860, height=900)


Launching India Cricket AI Selector...
  Model  : llama-3.1-8b-instant
  Engine : Pure Groq SDK — no openai/litellm/crewai
  Agents : 5 custom agents
  Players: 17
  Source : https://www.bcci.tv/
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d02b4e91ea17dcec5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[  8%] Agent 1 — Performance Analyst ranking players...
  [Agent 1] Performance Analyst...
[ 28%] Agent 2 — Conditions Specialist assessing pitch & weather...
  [Agent 2] Conditions Specialist...
[ 48%] Agent 3 — Fitness Monitor reviewing player readiness...
  [Agent 3] Fitness Monitor...
[ 66%] Agent 4 — Team Strategist proposing balanced XI...
  [Agent 4] Team Strategist...
[ 84%] Agent 5 — Chief Selector announcing final XI...
  [Agent 5] Chief Selector...
[100%] Complete! 15 players analysed by 5 agents.
